# 00 - Naive RAG and naive Q&A generation

Steps in this notebook:

1. Setup and model access check
2. Document preparation
3. Chunking
4. Indexing
5. Naive RAG
6. Naive Q&A generation
7. First evaluation

See [README.md](./README.md) for the full design.

## Step 1 - Setup

In [10]:
import logging
import os
import random
from pathlib import Path

from aieng.syn_data.text import (
    LLMSettings,
    OpenAICompatibleClient,
    load_implementation_dotenv,
    use_repo_root,
)
from rich.console import Console
from rich.table import Table


repo_root = use_repo_root()
load_implementation_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
console = Console(width=100)
print(f"Running from repository root: {repo_root}")

Running from repository root: /home/coder/CIBC1


In [11]:
SEED = 42
random.seed(SEED)

SOURCE_DOCS_DIR = repo_root / "implementations/qa_text_generation/data/documents"
DESIGN_DIR = repo_root / "implementations/RAG_QA_design"
DATA_DIR = DESIGN_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = DATA_DIR / "chunks.jsonl"
NAIVE_QA_PATH = DATA_DIR / "naive_qa.jsonl"
NAIVE_RESULTS_PATH = DATA_DIR / "naive_results.jsonl"

console.print(f"Source documents: {SOURCE_DOCS_DIR}")
console.print(f"Output data dir:  {DATA_DIR}")

Source documents: /home/coder/CIBC1/implementations/qa_text_generation/data/documents

Output data dir:  /home/coder/CIBC1/implementations/RAG_QA_design/data

### Model access check

The Vector proxy key only permits some models, so we send a tiny request to each candidate and see which ones answer.
Edit `CANDIDATE_MODELS` to test other names.

In [12]:
CANDIDATE_MODELS = [
    os.getenv("TEACHER_MODEL", "gpt-4o"),
    os.getenv("JUDGE_MODEL", "gemini-3.5-flash"),
    "gpt-4o-mini",
]
CANDIDATE_MODELS = list(dict.fromkeys(CANDIDATE_MODELS))


def check_model_access(model: str) -> tuple[bool, str]:
    """Send a tiny request and report whether the proxy allows the model (reasoning models need token headroom)."""
    client = OpenAICompatibleClient(LLMSettings.from_env(model_var="__UNSET__", default_model=model))
    try:
        reply = client.complete("Reply with the single word: ok", temperature=0.0, max_tokens=256)
    except Exception as exc:  # noqa: BLE001
        return False, str(exc)[:80]
    return True, reply.strip()[:40]


table = Table(show_header=True, header_style="bold magenta")
table.add_column("model")
table.add_column("allowed")
table.add_column("reply / error")

access = {}
for model in CANDIDATE_MODELS:
    ok, detail = check_model_access(model)
    access[model] = ok
    table.add_row(model, "yes" if ok else "NO", detail)

console.print(table)

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ model            ┃ allowed ┃ reply / error                                                       ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gpt-4o           │ yes     │ Ok                                                                  │
│ gemini-3.5-flash │ yes     │ ok                                                                  │
│ gpt-4o-mini      │ NO      │ 403 Client Error: Forbidden for url:                                │
│                  │         │ https://proxy.vectorinstitute.ai/v1/chat/co                         │
└──────────────────┴─────────┴─────────────────────────────────────────────────────────────────────┘

## Step 2 - Document preparation

The two source documents come from `qa_text_generation`:

- **CFPB credit card agreement**: in scope, this is what the RAG agent answers from.
- **SEC investor bulletin**: out of scope, used later for out-of-boundary questions.

The CFPB file is hard-wrapped at about 120 characters, so sentences are split across lines. The cleaner joins the wrapped lines, keeps list items on their own lines, and removes page numbers and repeated header/footer lines if there are any.

In [13]:
import re
from collections import Counter

from aieng.syn_data.text import list_domain_documents
from aieng.syn_data.text.documents import load_document_text


specs = {spec.doc_id: spec for spec in list_domain_documents("finance")}
raw_texts = {doc_id: load_document_text(Path(spec.local_path)) for doc_id, spec in specs.items()}

table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "role", "chars", "words", "lines"):
    table.add_column(column)
for doc_id, text in raw_texts.items():
    table.add_row(
        doc_id,
        str(specs[doc_id].role),
        str(len(text)),
        str(len(text.split())),
        str(text.count("\n") + 1),
    )
console.print(table)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ doc_id                     ┃ role           ┃ chars ┃ words ┃ lines ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ cfpb_credit_card_agreement │ policy_dense   │ 32027 │ 5541  │ 352   │
│ sec_investor_bulletin      │ scope_boundary │ 13795 │ 2133  │ 73    │
└────────────────────────────┴────────────────┴───────┴───────┴───────┘

In [14]:
PAGE_NUMBER_RE = re.compile(r"^\s*(?:page\s+)?\d+(?:\s+(?:of|/)\s+\d+)?\s*$", re.IGNORECASE)
LIST_ITEM_RE = re.compile(r"^(?:[a-z]\.\s|-\s|\d+\.\s)")  # "a. ", "- ", "1. "
SENTENCE_END = (".", "?", "!", ":", ";", '"', "\u201d", ")")
MAX_HEADER_LEN = 60  # a short line repeated 3+ times is treated as a header or footer


def clean_text(text: str) -> tuple[str, dict[str, int]]:
    """Join hard-wrapped lines, keep list items separate, and drop page numbers and repeated headers/footers."""
    lines = [line.strip() for line in text.split("\n")]
    counts = Counter(line for line in lines if line)
    repeated = {line for line, n in counts.items() if n >= 3 and len(line) <= MAX_HEADER_LEN}

    lengths = sorted(len(line) for line in lines if line)
    wrap_width = lengths[int(len(lengths) * 0.9)]  # the typical full-line width of the source

    stats = {"page_numbers_removed": 0, "headers_footers_removed": 0, "lines_joined": 0}
    blocks: list[str] = []
    for block in re.split(r"\n\s*\n", text):
        logical: list[str] = []
        current = ""
        last_len = 0
        for line in (raw.strip() for raw in block.split("\n")):
            if not line:
                continue
            if PAGE_NUMBER_RE.match(line):
                stats["page_numbers_removed"] += 1
                continue
            if line in repeated:
                stats["headers_footers_removed"] += 1
                continue
            if not current:
                current, last_len = line, len(line)
                continue
            starts_new_line = (
                LIST_ITEM_RE.match(line) is not None
                or (current.endswith(SENTENCE_END) and last_len < 0.85 * wrap_width)
                or (last_len < 0.6 * wrap_width and line[0].isupper())  # short line, then a capital: heading
            )
            if starts_new_line:
                logical.append(current)
                current = line
            else:
                current = f"{current} {line}"
                stats["lines_joined"] += 1
            last_len = len(line)
        if current:
            logical.append(current)
        if logical:
            blocks.append("\n".join(logical))
    return "\n\n".join(blocks), stats

In [15]:
clean_texts: dict[str, str] = {}
table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "chars before", "chars after", "lines joined", "page numbers", "headers/footers"):
    table.add_column(column)

for doc_id, text in raw_texts.items():
    cleaned, stats = clean_text(text)
    clean_texts[doc_id] = cleaned
    table.add_row(
        doc_id,
        str(len(text)),
        str(len(cleaned)),
        str(stats["lines_joined"]),
        str(stats["page_numbers_removed"]),
        str(stats["headers_footers_removed"]),
    )
console.print(table)

sample_doc = "cfpb_credit_card_agreement"
marker = "12. LIABILITY"
start = raw_texts[sample_doc].index(marker)
console.print("[bold]Before (section 12):[/bold]")
console.print(raw_texts[sample_doc][start : start + 450])
start = clean_texts[sample_doc].index(marker)
console.print("\n[bold]After (section 12):[/bold]")
console.print(clean_texts[sample_doc][start : start + 450])

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ doc_id              ┃ chars before ┃ chars after ┃ lines joined ┃ page numbers ┃ headers/footers ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ cfpb_credit_card_a… │ 32027        │ 32027       │ 225          │ 0            │ 0               │
│ sec_investor_bulle… │ 13795        │ 13795       │ 2            │ 0            │ 0               │
└─────────────────────┴──────────────┴─────────────┴──────────────┴──────────────┴─────────────────┘

Before (section 12):

12. LIABILITY FOR UNAUTHORIZED USE-LOST/STOLEN CARD NOTIFICATION — You agree to notify us
immediately, orally or in writing, at 630 32nd St. SE, Grand Rapids, MI 49548 or telephone (800) 
442-4757 seven (7)
days a week, 24 hours a day, of the loss, theft, or unauthorized use of your credit card. You may be
liable for the
unauthorized use of your credit card. You will not be liable for unauthorized use that occurs after 
you notify us of the loss,
t

After (section 12):

12. LIABILITY FOR UNAUTHORIZED USE-LOST/STOLEN CARD NOTIFICATION — You agree to notify us 
immediately, orally or in writing, at 630 32nd St. SE, Grand Rapids, MI 49548 or telephone (800) 
442-4757 seven (7) days a week, 24 hours a day, of the loss, theft, or unauthorized use of your 
credit card. You may be liable for the unauthorized use of your credit card. You will not be liable 
for unauthorized use that occurs after you notify us of the loss, t

## Step 3 - Chunking

This step reuses the existing `qa_text_generation` chunking, applied to the cleaned text:

- `paragraphs_from_document` splits each document on blank lines and merges pieces shorter than `min_chars` into their neighbour.
- `sample_test_paragraphs` holds out `DEFAULT_TEST_PARAS_PER_DOC` chunks per document, by chunk, with a fixed seed.
- Chunk IDs have the form `doc_id::pNNNN`.

Nothing is trained in this project, so the two splits are **development** (diagnose failures and tune) and **test** (frozen, run once at the end).
The package's `ParagraphSplit` enum only has `train` and `test`, so in memory `ParagraphSplit.TRAIN` stands for development. The saved file uses the names `development` and `test`.

Chunks are saved to `data/chunks.jsonl`, one JSON object per line.

In [16]:
from aieng.syn_data.text import (
    DEFAULT_TEST_PARAS_PER_DOC,
    Paragraph,
    ParagraphSplit,
    load_typed_jsonl,
    save_typed_jsonl,
)
from aieng.syn_data.text.documents import paragraphs_from_document, sample_test_paragraphs


MIN_CHARS = 80  # same value the original build_paragraph_splits uses
DEVELOPMENT = ParagraphSplit.TRAIN  # in memory, TRAIN means development; the file says "development"


def chunk_to_record(chunk: Paragraph) -> dict:
    record = chunk.to_dict()
    record["split"] = "development" if chunk.split is DEVELOPMENT else "test"
    return record


def chunk_from_record(record: dict) -> Paragraph:
    split = ParagraphSplit.TEST.value if record["split"] == "test" else DEVELOPMENT.value
    return Paragraph.from_dict({**record, "split": split})

chunks: list[Paragraph] = []
for doc_id, spec in specs.items():
    doc_chunks = paragraphs_from_document(spec, clean_texts[doc_id], min_chars=MIN_CHARS)
    test_chunks, validation_chunks = sample_test_paragraphs(doc_chunks, n_test=DEFAULT_TEST_PARAS_PER_DOC, seed=SEED)
    chunks.extend(test_chunks + validation_chunks)

chunks.sort(key=lambda chunk: (chunk.doc_id, chunk.index))

save_typed_jsonl(CHUNKS_PATH, chunks, to_dict=chunk_to_record)
console.print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

Saved 56 chunks to /home/coder/CIBC1/implementations/RAG_QA_design/data/chunks.jsonl

In [17]:
# Reload from disk to confirm the saved file round-trips.
chunks = load_typed_jsonl(CHUNKS_PATH, chunk_from_record)

table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "chunks", "test", "development", "min chars", "mean chars", "max chars"):
    table.add_column(column)
for doc_id in specs:
    doc = [chunk for chunk in chunks if chunk.doc_id == doc_id]
    sizes = [len(chunk.text) for chunk in doc]
    table.add_row(
        doc_id,
        str(len(doc)),
        str(sum(chunk.split is ParagraphSplit.TEST for chunk in doc)),
        str(sum(chunk.split is DEVELOPMENT for chunk in doc)),
        str(min(sizes)),
        str(round(sum(sizes) / len(sizes))),
        str(max(sizes)),
    )
console.print(table)

for chunk in chunks[:2]:
    console.print(f"\n[bold]{chunk.para_id}[/bold]  role={chunk.role.value}  split={'development' if chunk.split is DEVELOPMENT else 'test'}  index={chunk.index}  chars={len(chunk.text)}")
    console.print(chunk.text)

console.print("\n[bold]Raw line in chunks.jsonl:[/bold]")
console.print(CHUNKS_PATH.read_text(encoding="utf-8").splitlines()[0][:400] + " ...")

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ doc_id                     ┃ chunks ┃ test ┃ development ┃ min chars ┃ mean chars ┃ max chars ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ cfpb_credit_card_agreement │ 35     │ 10   │ 25          │ 99        │ 913        │ 4376      │
│ sec_investor_bulletin      │ 21     │ 10   │ 11          │ 244       │ 655        │ 1581      │
└────────────────────────────┴────────┴──────┴─────────────┴───────────┴────────────┴───────────┘

cfpb_credit_card_agreement::p0000  role=policy_dense  split=development  index=0  chars=755

VISA CARD
CONSUMER CREDIT CARD AGREEMENT
In this Agreement, “Agreement” means this Consumer Credit Card Agreement. “Disclosure” means the 
Credit Card Account Opening Disclosure. The Account Opening Disclosure is incorporated into this 
Consumer Credit Card Agreement and is part of the Agreement. In this Agreement the words "you," 
“your,” and "yours" mean each and all of those who agree to be bound by this Agreement; "card" means
the Visa credit card and any duplicates, renewals, or substitutions the Credit Union issues to you; 
"account" means your Visa credit card line-of-credit account with the Credit Union; and "Credit 
Union" means the Credit Union whose name appears on this Agreement or anyone to whom the Credit 
Union transfers this Agreement.

cfpb_credit_card_agreement::p0001  role=policy_dense  split=development  index=1  chars=3284

1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a line 
of credit for you. You agree that your credit limit is the maximum amount (purchases, cash advances,
finance charges, plus "other charges") which you will have outstanding on your account at any time. 
Unless disclosed otherwise, the Credit Union will not allow advances over the credit limit. If the 
Credit Union has a program whereby it allows payment of advances that exceed your credit limit, 
subject to a fee, the Credit Union will provide you with notice, either orally, in writing, or 
electronically (notwithstanding the requirements of the paragraph entitled “Statements and Notices”)
explaining your right to opt in to the Credit Union’s program whereby it will honor advance requests
over the credit limit. In the event you opt in to such a program, you agree to the terms of such a 
program. You may request an increase in your credit limit only by a method acceptable to the Credit 
Union. The Credit Union may increase or decrease your credit limit, refuse to make an advance and/or
terminate your account at any time for any reason not prohibited by law. If you are permitted to 
obtain cash advances on your account, we may, from time to time, issue convenience checks to you 
that may be drawn on your account. Convenience checks may not be used to make a payment on your 
account balance. If you use a convenience check, it will be posted to your account as a cash 
advance. We reserve the right to refuse to pay a convenience check drawn on your account for any 
reason and such refusal shall not constitute wrongful dishonor.
You may request that we stop the payment of a convenience check drawn on your account. You agree to 
pay any fee imposed to stop a payment on a convenience check issued on your account. You may make a 
stop payment request orally, if permitted, or in writing. Your request must be made with sufficient 
time in advance of the presentment of the check for payment to give us a reasonable opportunity to 
act on your request. In addition, your request must accurately describe the check including the 
exact account number, the payee, any check number that may be applicable, and the exact amount of 
the check. If permitted, you may make a stop payment request orally but such a request will expire 
after 14 days unless you confirm your request in writing within that time. Written stop payment 
orders are effective only for six months and may be renewed for additional six month periods by 
requesting in writing that the stop payment order be renewed. We are not required to notify you when
a stop payment order expires. If we re-credit your account after paying a check or draft over a 
valid and timely stop payment order, you agree to sign a statement describing the dispute with the 
payee, to assign to us all of your rights against the payee or other holders of the check or draft 
and to assist us in any legal action. You agree to indemnify and hold us harmless from all costs and
expenses, including attorney's fees, damages, or claims, related to our honoring your stop payment 
request or in failing to stop payment of an item as a result of incorrect information provided to us
or the giving of inadequate time to act upon a stop payment request.

Raw line in chunks.jsonl:

{"doc_id": "cfpb_credit_card_agreement", "para_id": "cfpb_credit_card_agreement::p0000", "text": 
"VISA CARD\nCONSUMER CREDIT CARD AGREEMENT\nIn this Agreement, “Agreement” means this Consumer 
Credit Card Agreement. “Disclosure” means the Credit Card Account Opening Disclosure. The Account 
Opening Disclosure is incorporated into this Consumer Credit Card Agreement and is part of the 
Agreement. In t ...

## Step 4 - Indexing

The retriever is the package's TF-IDF scoring (`tokenize`, `build_tfidf_index`, `tfidf_vector`, `cosine_similarity`). `retrieve_paragraphs` rebuilds the index on every call, so `TfidfIndex` builds the vectors once and reuses them.

- **Indexed:** all CFPB chunks, development and test together. The split only decides which chunks questions are generated from, so test questions can still find their source chunk.
- **Not indexed:** the SEC chunks. They are the source of out-of-scope questions, which should have nothing relevant to retrieve.
- The index is cheap to build, so it is rebuilt in memory and not saved.

In [ ]:
from aieng.syn_data.text.rag import (
    build_tfidf_index,
    cosine_similarity,
    retrieve_paragraphs,
    tfidf_vector,
    tokenize,
)
from aieng.syn_data.text.schemas import DocumentRole


class TfidfIndex:
    """TF-IDF index over chunks. Vectors are computed once; search ranks chunks by cosine similarity."""

    def __init__(self, chunks: list[Paragraph]) -> None:
        self.chunks = list(chunks)
        term_counters, self.idf = build_tfidf_index(self.chunks)
        self.vectors = [tfidf_vector(counts, self.idf) for counts in term_counters]

    def search(self, query: str, top_k: int = 3) -> list[tuple[float, Paragraph]]:
        """Return the top_k (score, chunk) pairs, best first."""
        query_vector = tfidf_vector(Counter(tokenize(query)), self.idf)
        scored = [(cosine_similarity(query_vector, vector), chunk) for vector, chunk in zip(self.vectors, self.chunks, strict=True)]
        scored.sort(key=lambda item: item[0], reverse=True)
        return scored[:top_k]


indexed_chunks = [chunk for chunk in chunks if chunk.role is DocumentRole.POLICY_DENSE]
index = TfidfIndex(indexed_chunks)
console.print(f"Indexed {len(indexed_chunks)} chunks, vocabulary size {len(index.idf)}")

In [ ]:
SANITY_QUERIES = [
    "What is the late payment fee?",
    "When will I not be charged a finance charge on new purchases?",
    "How do I dispute a billing error on my statement?",
    "What happens if I am in default?",
    "How should I protect my online investment account from fraud?",
]

# The cached index must return the same chunks as the package's retrieve_paragraphs.
for query in SANITY_QUERIES:
    cached = [chunk.para_id for _, chunk in index.search(query, top_k=3)]
    reference = [chunk.para_id for chunk in retrieve_paragraphs(query, indexed_chunks, top_k=3)]
    assert cached == reference, (query, cached, reference)
console.print("Cached index matches retrieve_paragraphs on all sanity queries.")

table = Table(show_header=True, header_style="bold magenta")
for column in ("query", "rank", "chunk", "split", "score", "preview"):
    table.add_column(column)
for query in SANITY_QUERIES:
    for rank, (score, chunk) in enumerate(index.search(query, top_k=3), start=1):
        table.add_row(
            query if rank == 1 else "",
            str(rank),
            chunk.para_id.split("::")[1],
            "development" if chunk.split is DEVELOPMENT else "test",
            f"{score:.3f}",
            chunk.text[:60].replace("\n", " "),
        )
console.print(table)